## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
import sys

sys.setrecursionlimit(300000)

def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return

    idx = 0
    m = data[idx]
    idx += 1

    s = data[idx]
    idx += 1

    t = data[idx]
    idx += 1

    a = data[idx:idx + m]
    res = []

    z = 0

    def op0():
        res.append(0)

        for i in range(m):
            if a[i] == s:
                a[i] = t
            elif a[i] == t:
                a[i] = s

    def op2(w):
        w %= m
        if w < 0:
            w += m

        if w == 0:
            return

        res.append(w)

        for i in range(m):
            a[i] = (a[i] + w) % m

    def op1(w):
        if w == 0:
            return

        res.append(-w)

        for i in range(m):
            a[i] ^= w

    def find_pair(x, y):
        d = (y - x + m - z + m) % m

        l = 0
        r = 0
        step = m // 2

        while step >= 2 * z:
            if d >= step:
                d -= step
                r += step // 2
            else:
                l += step // 2

            step >>= 1

        l += m // 2
        l += x & (z - 1)
        r += x & (z - 1)

        return l, r

    def change_pos(x, y):
        if (x // z) % 2 == (y // z) % 2:
            if (x // z) % 2 == 0:
                mid = (x & (z - 1)) + z
            else:
                mid = x & (z - 1)

            change_pos(x, mid)
            change_pos(y, mid)
            change_pos(x, mid)
            return

        p1, _ = find_pair(s, t)
        p2, _ = find_pair(x, y)

        op2((p2 - x + m) % m)
        op1(p2 ^ p1)
        op2((s - p1 + m) % m)

        op0()

        op2((p1 - s + m) % m)
        op1(p2 ^ p1)
        op2((x - p2 + m) % m)

    def build(b, length):
        mark = [0] * length

        for x in b:
            if x >= length:
                return False, []
            mark[x] = 1

        for x in mark:
            if not x:
                return False, []

        if length == 1:
            return True, []

        left = [0] * (length // 2)
        right = [0] * (length // 2)

        for i in range(length // 2):
            left[i] = b[i * 2] // 2
            right[i] = b[i * 2 + 1] // 2

        ok1, v_left = build(left, length // 2)
        ok2, v_right = build(right, length // 2)

        if not ok1 or not ok2:
            return False, []

        cur = []

        if b[0] & 1:
            if length == 2:
                cur.append(1)
            else:
                cur.append(-1)

        q1 = 0

        for x in v_left:
            if x > 0:
                cur.append(-1)
                cur.append(1)
            else:
                cur.append(x * 2)
                q1 ^= (-x) * 2

        if q1 != 0:
            cur.append(-q1)

        q2 = 0

        for x in v_right:
            if x > 0:
                cur.append(1)
                cur.append(-1)
            else:
                cur.append(x * 2)
                q2 ^= (-x) * 2

        if (q2 & (length // 2)) != (q1 & (length // 2)):
            return False, []

        if q1 >= length // 2:
            q1 -= length // 2

        if q2 >= length // 2:
            q2 -= length // 2

        if q1 != q2:
            return False, []

        ans = []

        for x in cur:
            if not ans:
                ans.append(x)
            else:
                if x < 0 and ans[-1] < 0:
                    ans[-1] = -((-ans[-1]) ^ (-x))

                    if ans[-1] == 0:
                        ans.pop()
                else:
                    ans.append(x)

        return True, ans

    z = (s - t + m) % m
    z &= -z

    if z == 0:
        z = m

    if z > 1:
        tmp = [0] * z

        for i in range(z):
            tmp[i] = a[i] & (z - 1)

        ok, ops = build(tmp, z)

        if not ok:
            print(-1)
            return

        for x in ops:
            if x > 0:
                op2(x)
            else:
                op1(-x)

    for r in range(z):
        b = []

        for i in range(r, m, z):
            b.append(a[i])

        b.sort()

        k = 0
        ok = True

        for i in range(r, m, z):
            if b[k] != i:
                ok = False
                break
            k += 1

        if not ok:
            print(-1)
            return

        for i in range(r, m, z):
            if a[i] != i:
                change_pos(i, a[i])

    out = [str(len(res))]

    for x in res:
        if x == 0:
            out.append("0")
        elif x < 0:
            out.append(f"1 {-x}")
        else:
            out.append(f"2 {x}")

    sys.stdout.write("\n".join(out))


if __name__ == "__main__":
    main()

## B 长跑

In [ ]:
import sys
import heapq

points = []


def check(total, target, limit, budget):
    arr = [(item[0], item[1]) for item in points]
    arr.sort()

    while arr and arr[-1][0] > target:
        arr.pop()

    total = len(arr)

    if target <= limit:
        return True

    inf = 1 << 60
    cost = [inf] * total

    for idx in range(total):
        if arr[idx][0] <= limit:
            cost[idx] = min(cost[idx], arr[idx][1])
        else:
            break

    heap = []

    for idx in range(total):
        if cost[idx] < inf:
            heapq.heappush(heap, (cost[idx], idx))

    while heap:
        cur_cost, cur_idx = heapq.heappop(heap)

        if cur_cost != cost[cur_idx]:
            continue

        if cur_cost > budget:
            continue

        cur_pos = arr[cur_idx][0]

        if target - cur_pos <= limit:
            return True

        for nxt_idx in range(cur_idx + 1, total):
            if arr[nxt_idx][0] - cur_pos > limit:
                break

            new_cost = cur_cost + arr[nxt_idx][1]

            if new_cost < cost[nxt_idx]:
                cost[nxt_idx] = new_cost
                heapq.heappush(heap, (new_cost, nxt_idx))

    return False


def main():
    global points

    nums = list(map(int, sys.stdin.buffer.read().split()))
    pos = 0
    output = []

    while pos + 4 <= len(nums):
        count = nums[pos]
        target = nums[pos + 1]
        step = nums[pos + 2]
        money = nums[pos + 3]
        pos += 4

        points = []

        for _ in range(count):
            place = nums[pos]
            price = nums[pos + 1]
            pos += 2
            points.append([place, price])

        output.append("Yes" if check(count, target, step, money) else "No")

    sys.stdout.write("\n".join(output))


if __name__ == "__main__":
    main()

## C 最长回文

In [ ]:
import sys

MOD_BASE = 911382323
MOD_MASK = (1 << 64) - 1


def get_radius(text):
    length = len(text)
    limit = length * 2 + 2

    arr = bytearray(limit + 1)
    arr[0] = 1
    arr[1] = 2

    write_pos = 2
    for value in text:
        arr[write_pos] = value
        write_pos += 1
        arr[write_pos] = 2
        write_pos += 1

    arr[limit] = 3

    radius = [0] * (limit + 1)
    right_edge = 0
    center = 0

    for pos in range(1, limit):
        if right_edge > pos:
            mirror = radius[2 * center - pos]
            bound = right_edge - pos
            radius[pos] = mirror if mirror < bound else bound
        else:
            radius[pos] = 1

        cur = radius[pos]

        while arr[pos + cur] == arr[pos - cur]:
            cur += 1

        radius[pos] = cur

        if pos + cur > right_edge:
            right_edge = pos + cur
            center = pos

    return radius, limit


def main():
    values = sys.stdin.buffer.read().split()
    if not values:
        return

    size = int(values[0])
    left_text = values[1]
    right_text = values[2]

    left_radius, limit = get_radius(left_text)
    right_radius, _ = get_radius(right_text)

    reversed_left = left_text[::-1]

    hash_left = [0] * (size + 1)
    hash_right = [0] * (size + 1)
    powers = [1] * (size + 1)

    base = MOD_BASE
    mask = MOD_MASK

    for pos in range(size):
        hash_left[pos + 1] = (hash_left[pos] * base + reversed_left[pos] + 7) & mask
        hash_right[pos + 1] = (hash_right[pos] * base + right_text[pos] + 7) & mask
        powers[pos + 1] = (powers[pos] * base) & mask

    answer = 1

    for center in range(2, limit + 1):
        span = left_radius[center]

        if right_radius[center - 2] > span:
            span = right_radius[center - 2]

        left_bound = center - span
        right_bound = center - 2 + span

        add_len = 0

        if 0 <= left_bound and right_bound < limit:
            if left_bound & 1:
                add_len = 1

                p1 = (left_bound - 3) // 2
                p2 = (right_bound - 1) // 2

                if p1 >= 0 and p2 < size and left_text[p1] == right_text[p2]:
                    upper = p1 + 1

                    if size - p2 < upper:
                        upper = size - p2

                    low = 1
                    high = upper
                    start = size - 1 - p1

                    while low < high:
                        mid = (low + high + 1) >> 1

                        h1 = (hash_left[start + mid] - hash_left[start] * powers[mid]) & mask
                        h2 = (hash_right[p2 + mid] - hash_right[p2] * powers[mid]) & mask

                        if h1 == h2:
                            low = mid
                        else:
                            high = mid - 1

                    add_len += low * 2

            elif left_bound >= 2 and right_bound >= 2:
                p1 = left_bound // 2 - 1
                p2 = right_bound // 2 - 1

                if p1 >= 0 and p2 < size and left_text[p1] == right_text[p2]:
                    upper = p1 + 1

                    if size - p2 < upper:
                        upper = size - p2

                    low = 1
                    high = upper
                    start = size - 1 - p1

                    while low < high:
                        mid = (low + high + 1) >> 1

                        h1 = (hash_left[start + mid] - hash_left[start] * powers[mid]) & mask
                        h2 = (hash_right[p2 + mid] - hash_right[p2] * powers[mid]) & mask

                        if h1 == h2:
                            low = mid
                        else:
                            high = mid - 1

                    add_len = low * 2

        current = span + add_len - 1

        if current > answer:
            answer = current

    print(answer)


if __name__ == "__main__":
    main()

## D 优惠券

In [ ]:
import sys

MAX_VALUE = 100000

status = [0] * (MAX_VALUE + 1)
last_seen = [0] * (MAX_VALUE + 1)


class Fenwick:
    def __init__(self, length):
        self.length = length
        self.tree = [0] * (length + 2)

    def add(self, index, delta):
        length = self.length
        tree = self.tree

        while index <= length:
            tree[index] += delta
            index += index & -index

    def prefix_sum(self, index):
        total = 0
        tree = self.tree

        while index > 0:
            total += tree[index]
            index -= index & -index

        return total

    def find_by_order(self, order):
        index = 0
        step = 1 << (self.length.bit_length() - 1)
        tree = self.tree

        while step:
            nxt = index + step

            if nxt <= self.length and tree[nxt] < order:
                index = nxt
                order -= tree[nxt]

            step >>= 1

        return index + 1


def main():
    tokens = sys.stdin.buffer.read().split()
    pointer = 0
    output = []

    while pointer < len(tokens):
        operation_count = int(tokens[pointer])
        pointer += 1

        bit = Fenwick(operation_count)
        touched = []
        answer = -1

        for position in range(1, operation_count + 1):
            command = tokens[pointer]
            pointer += 1

            if command == b'?':
                if answer == -1:
                    bit.add(position, 1)
                continue

            value = int(tokens[pointer])
            pointer += 1

            if answer != -1:
                continue

            if last_seen[value] == 0:
                touched.append(value)

            if command == b'I':
                status[value] += 1
            else:
                status[value] -= 1

            if status[value] < 0 or status[value] > 1:
                before_count = bit.prefix_sum(last_seen[value] - 1)
                total_count = bit.prefix_sum(operation_count)

                if total_count == before_count:
                    answer = position
                else:
                    remove_pos = bit.find_by_order(before_count + 1)
                    bit.add(remove_pos, -1)

                    if status[value] < 0:
                        status[value] = 0
                    else:
                        status[value] = 1

            last_seen[value] = position

        output.append(str(answer))

        for value in touched:
            status[value] = 0
            last_seen[value] = 0

    sys.stdout.write("\n".join(output))


if __name__ == "__main__":
    main()

## E 任意点

In [ ]:
import sys


class DisjointSet:
    def __init__(self, count):
        self.leader = list(range(count))

    def find_leader(self, index):
        while self.leader[index] != index:
            self.leader[index] = self.leader[self.leader[index]]
            index = self.leader[index]
        return index

    def merge(self, first, second):
        root_first = self.find_leader(first)
        root_second = self.find_leader(second)

        if root_first != root_second:
            self.leader[root_first] = root_second


def main():
    numbers = list(map(int, sys.stdin.buffer.read().split()))
    if not numbers:
        return

    point_count = numbers[0]
    coordinates = []
    position = 1

    for _ in range(point_count):
        row = numbers[position]
        col = numbers[position + 1]
        position += 2
        coordinates.append((row, col))

    dsu = DisjointSet(point_count)

    row_owner = {}
    col_owner = {}

    for index, (row, col) in enumerate(coordinates):
        if row in row_owner:
            dsu.merge(index, row_owner[row])
        else:
            row_owner[row] = index

        if col in col_owner:
            dsu.merge(index, col_owner[col])
        else:
            col_owner[col] = index

    component_count = 0

    for index in range(point_count):
        if dsu.find_leader(index) == index:
            component_count += 1

    print(component_count - 1)


if __name__ == "__main__":
    main()

## F 通配符匹配

In [ ]:
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
using namespace std;

using ull = unsigned long long;

static const ull HASH_BASE = 1315423911ull;

struct FixedBlock {
    int startOffset;
    string content;
    ull hashValue;
};

struct PatternSegment {
    int segmentLength = 0;
    vector<FixedBlock> fixedBlocks;
    int anchorIndex = -1;
    string anchorText;
    int anchorOffset = 0;
    vector<int> prefixTable;
};

ull calcHash(const string &text) {
    ull hashValue = 0;

    for (char ch : text) {
        hashValue = hashValue * HASH_BASE + (ull)(ch - 'a' + 1);
    }

    return hashValue;
}

vector<int> buildPrefixTable(const string &pattern) {
    vector<int> prefix(pattern.size(), 0);

    for (int i = 1, matched = 0; i < (int)pattern.size(); i++) {
        while (matched > 0 && pattern[i] != pattern[matched]) {
            matched = prefix[matched - 1];
        }

        if (pattern[i] == pattern[matched]) {
            matched++;
        }

        prefix[i] = matched;
    }

    return prefix;
}

vector<int> findOccurrences(const string &text, const string &pattern, const vector<int> &prefix) {
    vector<int> positions;

    if (pattern.empty()) {
        return positions;
    }

    for (int i = 0, matched = 0; i < (int)text.size(); i++) {
        while (matched > 0 && text[i] != pattern[matched]) {
            matched = prefix[matched - 1];
        }

        if (text[i] == pattern[matched]) {
            matched++;
        }

        if (matched == (int)pattern.size()) {
            positions.push_back(i - matched + 1);
            matched = prefix[matched - 1];
        }
    }

    return positions;
}

PatternSegment parseSegment(const string &segmentText) {
    PatternSegment segment;
    segment.segmentLength = (int)segmentText.size();

    int index = 0;

    while (index < (int)segmentText.size()) {
        if (segmentText[index] == '?') {
            index++;
            continue;
        }

        int nextIndex = index;

        while (nextIndex < (int)segmentText.size() && segmentText[nextIndex] != '?') {
            nextIndex++;
        }

        string blockText = segmentText.substr(index, nextIndex - index);
        segment.fixedBlocks.push_back({index, blockText, calcHash(blockText)});

        index = nextIndex;
    }

    for (int i = 0; i < (int)segment.fixedBlocks.size(); i++) {
        if (
            segment.anchorIndex == -1 ||
            segment.fixedBlocks[i].content.size() >
            segment.fixedBlocks[segment.anchorIndex].content.size()
        ) {
            segment.anchorIndex = i;
        }
    }

    if (segment.anchorIndex != -1) {
        segment.anchorText = segment.fixedBlocks[segment.anchorIndex].content;
        segment.anchorOffset = segment.fixedBlocks[segment.anchorIndex].startOffset;
        segment.prefixTable = buildPrefixTable(segment.anchorText);
    }

    return segment;
}

vector<ull> powerBase{1};

void preparePower(int length) {
    while ((int)powerBase.size() <= length) {
        powerBase.push_back(powerBase.back() * HASH_BASE);
    }
}

struct RollingHash {
    vector<ull> prefixHash;

    RollingHash(const string &text) {
        int length = (int)text.size();

        preparePower(length);

        prefixHash.assign(length + 1, 0);

        for (int i = 0; i < length; i++) {
            prefixHash[i + 1] = prefixHash[i] * HASH_BASE + (ull)(text[i] - 'a' + 1);
        }
    }

    ull getHash(int left, int right) const {
        return prefixHash[right] - prefixHash[left] * powerBase[right - left];
    }
};

bool matchSegmentAt(
    const PatternSegment &segment,
    const string &text,
    const RollingHash &hashTool,
    int startPosition
) {
    if (startPosition < 0 || startPosition + segment.segmentLength > (int)text.size()) {
        return false;
    }

    for (const auto &block : segment.fixedBlocks) {
        int left = startPosition + block.startOffset;
        int right = left + (int)block.content.size();

        if (hashTool.getHash(left, right) != block.hashValue) {
            return false;
        }
    }

    return true;
}

int findNextSegmentPosition(
    const PatternSegment &segment,
    const string &text,
    const RollingHash &hashTool,
    const vector<int> &anchorPositions,
    int searchStart,
    int searchLimit
) {
    if (segment.segmentLength == 0) {
        return searchStart;
    }

    if (segment.fixedBlocks.empty()) {
        return searchStart + segment.segmentLength <= searchLimit ? searchStart : -1;
    }

    int requiredAnchorPos = searchStart + segment.anchorOffset;
    auto iter = lower_bound(anchorPositions.begin(), anchorPositions.end(), requiredAnchorPos);

    while (iter != anchorPositions.end()) {
        int anchorPosition = *iter;
        int segmentStart = anchorPosition - segment.anchorOffset;

        if (segmentStart + segment.segmentLength > searchLimit) {
            break;
        }

        if (matchSegmentAt(segment, text, hashTool, segmentStart)) {
            return segmentStart;
        }

        iter++;
    }

    return -1;
}

bool matchText(
    const vector<PatternSegment> &segments,
    bool hasLeadingStar,
    bool hasTrailingStar,
    const string &text
) {
    RollingHash hashTool(text);

    int segmentCount = (int)segments.size();

    if (!hasLeadingStar && !hasTrailingStar && segmentCount == 1) {
        return (
            segments[0].segmentLength == (int)text.size() &&
            matchSegmentAt(segments[0], text, hashTool, 0)
        );
    }

    vector<vector<int>> anchorPositions(segmentCount);

    for (int i = 0; i < segmentCount; i++) {
        if (!segments[i].anchorText.empty()) {
            anchorPositions[i] = findOccurrences(
                text,
                segments[i].anchorText,
                segments[i].prefixTable
            );
        }
    }

    int leftBound = 0;
    int rightBound = (int)text.size();

    int leftSegment = 0;
    int rightSegment = segmentCount - 1;

    if (!hasLeadingStar) {
        if (!matchSegmentAt(segments[0], text, hashTool, 0)) {
            return false;
        }

        leftBound = segments[0].segmentLength;
        leftSegment = 1;
    }

    if (!hasTrailingStar) {
        int startPosition = (int)text.size() - segments[segmentCount - 1].segmentLength;

        if (startPosition < leftBound) {
            return false;
        }

        if (!matchSegmentAt(segments[segmentCount - 1], text, hashTool, startPosition)) {
            return false;
        }

        rightBound = startPosition;
        rightSegment = segmentCount - 2;
    }

    int currentPosition = leftBound;

    for (int i = leftSegment; i <= rightSegment; i++) {
        int segmentStart = findNextSegmentPosition(
            segments[i],
            text,
            hashTool,
            anchorPositions[i],
            currentPosition,
            rightBound
        );

        if (segmentStart == -1) {
            return false;
        }

        currentPosition = segmentStart + segments[i].segmentLength;
    }

    return true;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string rawPattern;
    cin >> rawPattern;

    string compactPattern;

    for (char ch : rawPattern) {
        if (ch == '*' && !compactPattern.empty() && compactPattern.back() == '*') {
            continue;
        }

        compactPattern.push_back(ch);
    }

    bool hasLeadingStar = !compactPattern.empty() && compactPattern.front() == '*';
    bool hasTrailingStar = !compactPattern.empty() && compactPattern.back() == '*';

    vector<PatternSegment> segments;
    string currentSegment;

    for (char ch : compactPattern) {
        if (ch == '*') {
            segments.push_back(parseSegment(currentSegment));
            currentSegment.clear();
        } else {
            currentSegment.push_back(ch);
        }
    }

    segments.push_back(parseSegment(currentSegment));

    int queryCount;
    cin >> queryCount;

    while (queryCount--) {
        string text;
        cin >> text;

        cout << (matchText(segments, hasLeadingStar, hasTrailingStar, text) ? "YES" : "NO") << '\n';
    }

    return 0;
}

## G 汉诺塔

In [ ]:
import sys


def get_index(ch):
    return ord(ch) - ord("A")


def main():
    data = sys.stdin.read().split()
    if not data:
        return

    disk_count = int(data[0])
    moves = data[1:7]

    order = [[0] * 3 for _ in range(3)]

    for i in range(6):
        x = get_index(moves[i][0])
        y = get_index(moves[i][1])
        order[x][y] = i

    target = [0] * 3

    for x in range(3):
        choices = []

        for y in range(3):
            if y != x:
                choices.append(y)

        first = choices[0]
        second = choices[1]

        if order[x][first] < order[x][second]:
            target[x] = first
        else:
            target[x] = second

    power3 = [1] * 31

    for i in range(1, 31):
        power3[i] = power3[i - 1] * 3

    start = 0

    if target[target[start]] == start:
        answer = 2 * power3[disk_count - 1] - 1
    elif target[target[target[start]]] == start:
        answer = (1 << disk_count) - 1
    else:
        answer = power3[disk_count - 1]

    print(answer)


if __name__ == "__main__":
    main()

## H 马步距离

In [ ]:
import sys


def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return

    x1, y1, x2, y2 = data

    a = abs(x1 - x2)
    b = abs(y1 - y2)

    if a < b:
        a, b = b, a

    if a == 1 and b == 0:
        print(3)
        return

    if a == 2 and b == 2:
        print(4)
        return

    ans = max((a + 1) // 2, (a + b + 2) // 3)

    while (ans + a + b) % 2 != 0:
        ans += 1

    print(ans)


if __name__ == "__main__":
    main()

## I 直方图最大矩形

In [ ]:
#
# 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
#
# 
# @param heights int整型一维数组 
# @return int整型
#
class Solution:
    def largestRectangleArea(self , heights: List[int]) -> int:
        n = len(heights)
        if n == 0:
            return 0

        stack = []
        ans = 0

        for i in range(n + 1):
            h = 0 if i == n else heights[i]

            while stack and heights[stack[-1]] > h:
                cur = stack.pop()
                left = -1 if not stack else stack[-1]
                width = i - left - 1
                ans = max(ans, heights[cur] * width)

            stack.append(i)

        return ans

## J 消防局的设立

In [ ]:
import sys


def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return

    size = data[0]
    parent = [0] * (size + 1)
    depth = [0] * (size + 1)

    idx = 1
    for node in range(2, size + 1):
        parent[node] = data[idx]
        idx += 1
        depth[node] = depth[parent[node]] + 1

    order = list(range(1, size + 1))
    order.sort(key=lambda x: depth[x], reverse=True)

    placed = [0] * (size + 1)
    child_mark = [0] * (size + 1)
    grand_mark = [0] * (size + 1)

    def is_safe(node):
        p = parent[node]
        gp = parent[p] if p else 0

        if placed[node]:
            return True
        if p and placed[p]:
            return True
        if gp and placed[gp]:
            return True
        if child_mark[node] > 0:
            return True
        if grand_mark[node] > 0:
            return True
        if p and child_mark[p] > 0:
            return True

        return False

    answer = 0

    for node in order:
        if depth[node] > 2 and not is_safe(node):
            target = parent[parent[node]]

            if not placed[target]:
                placed[target] = 1
                answer += 1

                p = parent[target]
                gp = parent[p] if p else 0

                if p:
                    child_mark[p] += 1
                if gp:
                    grand_mark[gp] += 1

    for node in range(1, size + 1):
        if not is_safe(node):
            answer += 1
            break

    print(answer)


if __name__ == "__main__":
    main()